# 18tA — Canonical HKO publication-availability panel

This stage freezes the information time at which each realised
HKO daily maximum may first enter a fitted post-processing model.

The canonical availability time is

\[
\tau^{\mathrm{HKO},*}(d)
=
\begin{cases}
\tau^{\mathrm{HKO},\mathrm{obs}}(d),
& \text{when a timestamp has been independently verified},\\
14{:}00\ \mathrm{HKT}
\text{ on the first analytical operational working day after }d,
& \text{otherwise}.
\end{cases}
\]

The fallback is an analytical embargo, not a claim about the exact
publication second. An analytical operational working day is
Monday to Friday excluding the gazetted 2026 Hong Kong general
holidays.

The official HKO service states that its climatological data are
updated every working day before 2 p.m., up to the previous day.
The holiday calendar is taken from the Hong Kong Government's
gazetted 2026 general-holiday list.

No final modelling split is assigned in this stage.

In [1]:
from __future__ import annotations

import hashlib
import json
import platform
import sys
from datetime import datetime, time, timezone
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run this notebook from the repository root, not {ROOT}"
    )

UTC = timezone.utc
HKT = ZoneInfo("Asia/Hong_Kong")
STEP = "18tA"

INPUT_DIR = (
    ROOT
    / "data/processed/18s_expanded_march_june_canonical_sample"
)
DATE_INVENTORY_PATH = (
    INPUT_DIR / "18s_expanded_date_inventory.csv"
)
TARGET_PATH = (
    INPUT_DIR
    / "18s_expanded_certified_contract_outcome_panel.csv"
)
INPUT_SUMMARY_PATH = (
    INPUT_DIR / "18s_expanded_sample_summary.json"
)
INPUT_MANIFEST_PATH = (
    INPUT_DIR / "18s_expanded_sha256_manifest.csv"
)

MANUAL_EVIDENCE_PATH = (
    ROOT
    / "data/manual/18t_hko_verified_publication_timestamps.csv"
)

OUT_DIR = (
    ROOT
    / "data/processed/18tA_hko_publication_availability"
)
REPORT_DIR = (
    ROOT
    / "reports/18tA_hko_publication_availability"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

OFFICIAL_SOURCES = {
    "hko_update_convention": {
        "title": "HKO Climatological Information Services",
        "url": "https://www.hko.gov.hk/en/cis/awsDailyElement.htm",
        "statement": (
            "Data is updated every working day before 2 p.m., "
            "up to the previous day."
        ),
    },
    "hk_general_holidays_2026": {
        "title": "GovHK: General holidays for 2026",
        "url": "https://www.gov.hk/en/about/abouthk/holiday/2026.htm",
        "statement": (
            "Gazetted list of Hong Kong general holidays for 2026."
        ),
    },
}

# All gazetted 2026 general holidays that can affect the sample's
# next-working-day calculation through 2 July 2026.
GAZETTED_HOLIDAYS = [
    ("2026-04-03", "Good Friday"),
    ("2026-04-04", "The day following Good Friday"),
    (
        "2026-04-06",
        "The day following Ching Ming Festival",
    ),
    (
        "2026-04-07",
        "The day following Easter Monday",
    ),
    ("2026-05-01", "Labour Day"),
    (
        "2026-05-25",
        "The day following the Birthday of the Buddha",
    ),
    ("2026-06-19", "Tuen Ng Festival"),
    (
        "2026-07-01",
        "Hong Kong Special Administrative Region "
        "Establishment Day",
    ),
]

EXPECTED_DATES = 103
EXPECTED_CONTRACTS = 1133

for required in [
    DATE_INVENTORY_PATH,
    TARGET_PATH,
    INPUT_SUMMARY_PATH,
    INPUT_MANIFEST_PATH,
]:
    if not required.is_file():
        raise FileNotFoundError(
            f"Required verified 18s input is missing: {required}"
        )

print(f"Repository root: {ROOT}")
print(f"Optional verified-timestamp file: {MANUAL_EVIDENCE_PATH}")

Repository root: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket
Optional verified-timestamp file: /Users/edwardlee/Desktop/2026MScWeatherForecastingPolymarket/data/manual/18t_hko_verified_publication_timestamps.csv


In [2]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(series: pd.Series, name: str) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
                "verified": True,
                "unverified": False,
            }
        )
    )

    if parsed.isna().any():
        bad = series.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse Boolean field {name}: {bad}"
        )

    return parsed.astype(bool)


def first_existing_column(
    frame: pd.DataFrame,
    aliases: list[str],
    *,
    required: bool = True,
) -> str | None:
    direct = set(frame.columns)
    normalised = {
        "".join(
            character
            for character in str(column).lower()
            if character.isalnum()
        ): column
        for column in frame.columns
    }

    for alias in aliases:
        if alias in direct:
            return alias

        key = "".join(
            character
            for character in alias.lower()
            if character.isalnum()
        )
        if key in normalised:
            return normalised[key]

    if required:
        raise KeyError(
            f"None of {aliases} found. "
            f"Columns={list(frame.columns)}"
        )

    return None


def parse_timestamp_to_hkt(
    series: pd.Series,
    *,
    field_name: str,
    assume_timezone: str | None,
) -> pd.Series:
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(
        text,
        errors="coerce",
    )

    if parsed.isna().any():
        bad = text.loc[
            parsed.isna()
        ].drop_duplicates().tolist()
        raise ValueError(
            f"Could not parse {field_name}: {bad}"
        )

    if getattr(parsed.dt, "tz", None) is None:
        if assume_timezone is None:
            raise ValueError(
                f"{field_name} is timezone-naive and no "
                "assumed timezone was supplied."
            )
        parsed = parsed.dt.tz_localize(
            assume_timezone,
            ambiguous="raise",
            nonexistent="raise",
        )
    else:
        parsed = parsed.dt.tz_convert(HKT)

    return parsed.dt.tz_convert(HKT)


holiday_table = pd.DataFrame(
    GAZETTED_HOLIDAYS,
    columns=["calendar_date", "holiday_name"],
)
holiday_table["calendar_date"] = pd.to_datetime(
    holiday_table["calendar_date"],
    errors="raise",
)
holiday_dates = set(
    holiday_table["calendar_date"]
    .dt.normalize()
    .tolist()
)


def is_operational_working_day(
    calendar_date: pd.Timestamp,
) -> bool:
    date = pd.Timestamp(calendar_date).normalize()
    return (
        date.weekday() < 5
        and date not in holiday_dates
    )


def first_operational_day_after(
    event_date: pd.Timestamp,
) -> pd.Timestamp:
    candidate = (
        pd.Timestamp(event_date).normalize()
        + pd.Timedelta(days=1)
    )

    for _ in range(15):
        if is_operational_working_day(candidate):
            return candidate
        candidate += pd.Timedelta(days=1)

    raise RuntimeError(
        f"No operational working day found after {event_date}"
    )

In [3]:
with INPUT_SUMMARY_PATH.open(encoding="utf-8") as handle:
    input_summary = json.load(handle)

if input_summary.get("verdict") != "PASS":
    raise AssertionError(
        f"18s input verdict is not PASS: "
        f"{input_summary.get('verdict')}"
    )

if int(input_summary.get("certified_dates", -1)) != EXPECTED_DATES:
    raise AssertionError(
        "18s input summary does not contain 103 dates."
    )

if int(
    input_summary.get("certified_contracts", -1)
) != EXPECTED_CONTRACTS:
    raise AssertionError(
        "18s input summary does not contain 1,133 contracts."
    )

date_inventory = pd.read_csv(
    DATE_INVENTORY_PATH,
    low_memory=False,
)
target = pd.read_csv(
    TARGET_PATH,
    dtype={"market_id": str},
    low_memory=False,
)

date_inventory["event_date"] = pd.to_datetime(
    date_inventory["event_date"],
    errors="raise",
)
target["event_date"] = pd.to_datetime(
    target["event_date"],
    errors="raise",
)

if len(date_inventory) != EXPECTED_DATES:
    raise AssertionError(
        f"Expected 103 date rows, found {len(date_inventory)}"
    )

if len(target) != EXPECTED_CONTRACTS:
    raise AssertionError(
        f"Expected 1,133 target rows, found {len(target)}"
    )

if date_inventory["event_date"].duplicated().any():
    raise AssertionError(
        "Duplicate dates in the 18s date inventory."
    )

if not target.groupby(
    "event_date"
)["market_id"].size().eq(11).all():
    raise AssertionError(
        "An 18s target date does not contain 11 contracts."
    )

daily_target = (
    target.groupby(
        ["event_date", "sample_block"],
        as_index=False,
    )
    .agg(
        hko_daily_max_c=("hko_daily_max_c", "first"),
        certified_contracts=("market_id", "size"),
        realised_winners=("Y_event_int", "sum"),
    )
)

if not daily_target["certified_contracts"].eq(11).all():
    raise AssertionError(
        "Daily target aggregation is not eleven contracts per date."
    )

if not daily_target["realised_winners"].eq(1).all():
    raise AssertionError(
        "Daily target aggregation lacks exactly one winner."
    )

availability = date_inventory[
    ["event_date", "sample_block"]
].merge(
    daily_target,
    on=["event_date", "sample_block"],
    how="left",
    validate="one_to_one",
)

print("Verified 18s date inputs: PASS")
print(f"Settlement dates: {len(availability):,}")

Verified 18s date inputs: PASS
Settlement dates: 103


In [4]:
evidence_columns = [
    "event_date",
    "observed_publication_timestamp_hkt",
    "verification_status",
    "evidence_reference",
    "evidence_note",
    "source_file",
]

evidence = pd.DataFrame(columns=evidence_columns)
evidence_file_status = "ABSENT"

if MANUAL_EVIDENCE_PATH.is_file():
    raw_evidence = pd.read_csv(
        MANUAL_EVIDENCE_PATH,
        low_memory=False,
    )

    event_date_column = first_existing_column(
        raw_evidence,
        [
            "event_date",
            "settlement_date",
            "hko_event_date",
            "date",
        ],
    )

    timestamp_hkt_column = first_existing_column(
        raw_evidence,
        [
            "verified_publication_timestamp_hkt",
            "observed_publication_timestamp_hkt",
            "hko_publication_timestamp_hkt",
            "publication_timestamp_hkt",
        ],
        required=False,
    )

    timestamp_utc_column = first_existing_column(
        raw_evidence,
        [
            "verified_publication_timestamp_utc",
            "observed_publication_timestamp_utc",
            "hko_publication_timestamp_utc",
            "publication_timestamp_utc",
        ],
        required=False,
    )

    if (
        timestamp_hkt_column is None
        and timestamp_utc_column is None
    ):
        raise KeyError(
            "The manual evidence file must contain a verified "
            "HKT or UTC publication timestamp column."
        )

    verification_column = first_existing_column(
        raw_evidence,
        [
            "verified",
            "verification_status",
            "timestamp_verified",
            "is_verified",
        ],
    )

    reference_column = first_existing_column(
        raw_evidence,
        [
            "evidence_reference",
            "evidence_url",
            "source_reference",
            "source_url",
        ],
    )

    note_column = first_existing_column(
        raw_evidence,
        [
            "evidence_note",
            "note",
            "verification_note",
        ],
        required=False,
    )

    verified = parse_bool(
        raw_evidence[verification_column],
        verification_column,
    )

    if timestamp_hkt_column is not None:
        parsed_timestamp = parse_timestamp_to_hkt(
            raw_evidence[timestamp_hkt_column],
            field_name=timestamp_hkt_column,
            assume_timezone="Asia/Hong_Kong",
        )
    else:
        parsed_timestamp = parse_timestamp_to_hkt(
            raw_evidence[timestamp_utc_column],
            field_name=timestamp_utc_column,
            assume_timezone="UTC",
        )

    evidence = pd.DataFrame(
        {
            "event_date": pd.to_datetime(
                raw_evidence[event_date_column],
                errors="raise",
            ).dt.normalize(),
            "observed_publication_timestamp_hkt": (
                parsed_timestamp
            ),
            "verification_status": verified,
            "evidence_reference": (
                raw_evidence[reference_column]
                .astype("string")
                .str.strip()
            ),
            "evidence_note": (
                raw_evidence[note_column].astype("string")
                if note_column is not None
                else pd.Series(
                    pd.NA,
                    index=raw_evidence.index,
                    dtype="string",
                )
            ),
            "source_file": str(
                MANUAL_EVIDENCE_PATH.relative_to(ROOT)
            ),
        }
    )

    evidence = evidence.loc[
        evidence["verification_status"]
    ].copy()

    if evidence[
        "evidence_reference"
    ].isna().any() or evidence[
        "evidence_reference"
    ].eq("").any():
        raise AssertionError(
            "Every verified observed timestamp requires a "
            "non-empty evidence reference."
        )

    if evidence["event_date"].duplicated().any():
        raise AssertionError(
            "The manual evidence file has duplicate verified dates."
        )

    unknown_dates = set(evidence["event_date"]) - set(
        availability["event_date"]
    )
    if unknown_dates:
        raise AssertionError(
            "Manual evidence contains dates outside the frozen "
            f"103-date sample: {sorted(unknown_dates)}"
        )

    evidence_file_status = "LOADED_AND_VERIFIED"

print(
    "Verified observed timestamp evidence status: "
    f"{evidence_file_status}"
)
print(f"Verified observed timestamps: {len(evidence):,}")

Verified observed timestamp evidence status: ABSENT
Verified observed timestamps: 0


In [5]:
fallback_dates = availability["event_date"].map(
    first_operational_day_after
)

availability["event_weekday"] = availability[
    "event_date"
].dt.day_name()
availability[
    "fallback_operational_date"
] = fallback_dates

fallback_naive = pd.to_datetime(
    fallback_dates.dt.strftime("%Y-%m-%d")
    + " 14:00:00",
    errors="raise",
)
availability[
    "fallback_publication_timestamp_hkt"
] = fallback_naive.dt.tz_localize(HKT)

calendar_start = availability[
    "event_date"
].min().normalize()
calendar_end = max(
    availability["event_date"].max().normalize(),
    availability[
        "fallback_operational_date"
    ].max().normalize(),
)

calendar = pd.DataFrame(
    {
        "calendar_date": pd.date_range(
            calendar_start,
            calendar_end,
            freq="D",
        )
    }
)
calendar["weekday_name"] = calendar[
    "calendar_date"
].dt.day_name()
calendar["is_weekend"] = calendar[
    "calendar_date"
].dt.weekday.ge(5)
calendar["is_gazetted_holiday"] = calendar[
    "calendar_date"
].isin(holiday_dates)
calendar = calendar.merge(
    holiday_table,
    on="calendar_date",
    how="left",
    validate="one_to_one",
)
calendar["is_operational_working_day"] = (
    ~calendar["is_weekend"]
    & ~calendar["is_gazetted_holiday"]
)
calendar[
    "analytical_operational_definition"
] = (
    "Monday-Friday excluding gazetted Hong Kong "
    "general holidays"
)

skipped_rows: list[dict[str, Any]] = []

for row in availability.itertuples(index=False):
    skipped_dates = pd.date_range(
        row.event_date + pd.Timedelta(days=1),
        row.fallback_operational_date
        - pd.Timedelta(days=1),
        freq="D",
    )

    skipped_calendar = calendar.loc[
        calendar["calendar_date"].isin(skipped_dates)
    ]

    skipped_rows.append(
        {
            "event_date": row.event_date,
            "fallback_operational_date": (
                row.fallback_operational_date
            ),
            "calendar_days_to_fallback": int(
                (
                    row.fallback_operational_date
                    - row.event_date
                ).days
            ),
            "weekend_dates_crossed": "|".join(
                skipped_calendar.loc[
                    skipped_calendar["is_weekend"],
                    "calendar_date",
                ]
                .dt.strftime("%Y-%m-%d")
                .tolist()
            ),
            "holiday_dates_crossed": "|".join(
                skipped_calendar.loc[
                    skipped_calendar[
                        "is_gazetted_holiday"
                    ],
                    "calendar_date",
                ]
                .dt.strftime("%Y-%m-%d")
                .tolist()
            ),
        }
    )

skipped = pd.DataFrame(skipped_rows)

availability = availability.merge(
    skipped,
    on=[
        "event_date",
        "fallback_operational_date",
    ],
    how="left",
    validate="one_to_one",
)

availability = availability.merge(
    evidence[
        [
            "event_date",
            "observed_publication_timestamp_hkt",
            "evidence_reference",
            "evidence_note",
            "source_file",
        ]
    ],
    on="event_date",
    how="left",
    validate="one_to_one",
)

availability[
    "observed_publication_timestamp_hkt"
] = pd.to_datetime(
    availability[
        "observed_publication_timestamp_hkt"
    ],
    utc=True,
    errors="coerce",
).dt.tz_convert(HKT)

observed_available = availability[
    "observed_publication_timestamp_hkt"
].notna()

availability["availability_source"] = np.where(
    observed_available,
    "VERIFIED_OBSERVED",
    "ANALYTICAL_FALLBACK",
)
availability[
    "exact_publication_timestamp_claim"
] = observed_available

availability[
    "hko_publication_available_hkt"
] = availability[
    "fallback_publication_timestamp_hkt"
].copy()
availability.loc[
    observed_available,
    "hko_publication_available_hkt",
] = availability.loc[
    observed_available,
    "observed_publication_timestamp_hkt",
]

availability[
    "hko_publication_available_hkt"
] = pd.to_datetime(
    availability[
        "hko_publication_available_hkt"
    ],
    utc=True,
    errors="raise",
).dt.tz_convert(HKT)

availability[
    "hko_publication_available_utc"
] = availability[
    "hko_publication_available_hkt"
].dt.tz_convert("UTC")

event_end_hkt = pd.to_datetime(
    availability["event_date"].dt.strftime("%Y-%m-%d")
    + " 23:59:59.999999",
    errors="raise",
).dt.tz_localize(HKT)

availability[
    "availability_lag_hours_after_event_end"
] = (
    availability[
        "hko_publication_available_hkt"
    ]
    - event_end_hkt
).dt.total_seconds() / 3600.0

availability[
    "availability_calendar_days_after_event"
] = (
    availability[
        "hko_publication_available_hkt"
    ].dt.tz_localize(None).dt.normalize()
    - availability["event_date"]
).dt.days

availability[
    "fallback_is_exact_timestamp_claim"
] = False
availability[
    "fallback_convention"
] = (
    "14:00 HKT on first Monday-Friday after event date "
    "excluding gazetted Hong Kong general holidays"
)
availability[
    "hko_service_update_statement"
] = OFFICIAL_SOURCES[
    "hko_update_convention"
]["statement"]
availability[
    "final_modelling_split"
] = "UNASSIGNED"

if len(availability) != EXPECTED_DATES:
    raise AssertionError(
        "Availability panel does not contain 103 rows."
    )

if availability["event_date"].duplicated().any():
    raise AssertionError(
        "Availability panel contains duplicate event dates."
    )

if not availability[
    "fallback_operational_date"
].map(is_operational_working_day).all():
    raise AssertionError(
        "A fallback date is not an operational working day."
    )

if not (
    availability["fallback_operational_date"]
    > availability["event_date"]
).all():
    raise AssertionError(
        "A fallback date is not after its event date."
    )

if not (
    availability[
        "hko_publication_available_hkt"
    ]
    > event_end_hkt
).all():
    raise AssertionError(
        "A canonical availability timestamp is not after "
        "the end of its settlement date."
    )

if (
    availability[
        "availability_calendar_days_after_event"
    ]
    > 10
).any():
    raise AssertionError(
        "A publication availability lag exceeds ten days."
    )

if not availability[
    "final_modelling_split"
].eq("UNASSIGNED").all():
    raise AssertionError(
        "A final modelling split was assigned prematurely."
    )

print("Canonical HKO availability construction: PASS")
print(
    "Verified observed rows: "
    f"{int(observed_available.sum()):,}"
)
print(
    "Analytical fallback rows: "
    f"{int((~observed_available).sum()):,}"
)
display(
    availability[
        [
            "event_date",
            "sample_block",
            "availability_source",
            "hko_publication_available_hkt",
            "calendar_days_to_fallback",
            "weekend_dates_crossed",
            "holiday_dates_crossed",
        ]
    ].head(15)
)

Canonical HKO availability construction: PASS
Verified observed rows: 0
Analytical fallback rows: 103


,event_date,sample_block,availability_source,hko_publication_available_hkt,calendar_days_to_fallback,weekend_dates_crossed,holiday_dates_crossed
0,2026-03-16,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-17 14:00:00+08:00,1,,
1,2026-03-17,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-18 14:00:00+08:00,1,,
2,2026-03-18,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-19 14:00:00+08:00,1,,
3,2026-03-19,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-20 14:00:00+08:00,1,,
4,2026-03-21,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-23 14:00:00+08:00,2,2026-03-22,
5,2026-03-22,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-23 14:00:00+08:00,1,,
6,2026-03-23,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-24 14:00:00+08:00,1,,
7,2026-03-24,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-25 14:00:00+08:00,1,,
8,2026-03-25,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-26 14:00:00+08:00,1,,
9,2026-03-26,march_may_baseline,ANALYTICAL_FALLBACK,2026-03-27 14:00:00+08:00,1,,


In [6]:
check_rows: list[dict[str, Any]] = []

def add_check(
    check: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": check,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "verified_18s_input_pass",
    input_summary.get("verdict") == "PASS",
    str(input_summary.get("verdict")),
)
add_check(
    "availability_rows_103",
    len(availability) == 103,
    f"rows={len(availability)}",
)
add_check(
    "availability_dates_unique",
    not availability["event_date"].duplicated().any(),
    "one row per frozen settlement date",
)
add_check(
    "fallback_dates_are_operational",
    availability[
        "fallback_operational_date"
    ].map(is_operational_working_day).all(),
    "Monday-Friday and not gazetted holiday",
)
add_check(
    "fallback_dates_follow_event_dates",
    (
        availability["fallback_operational_date"]
        > availability["event_date"]
    ).all(),
    "strictly later calendar date",
)
add_check(
    "canonical_availability_after_event_end",
    (
        availability[
            "availability_lag_hours_after_event_end"
        ]
        > 0
    ).all(),
    "no same-day outcome availability",
)
add_check(
    "availability_sources_exhaustive",
    availability[
        "availability_source"
    ].isin(
        [
            "VERIFIED_OBSERVED",
            "ANALYTICAL_FALLBACK",
        ]
    ).all(),
    "observed if verified, otherwise fallback",
)
add_check(
    "observed_plus_fallback_equals_103",
    (
        availability[
            "availability_source"
        ].eq("VERIFIED_OBSERVED").sum()
        + availability[
            "availability_source"
        ].eq("ANALYTICAL_FALLBACK").sum()
        == 103
    ),
    (
        "observed="
        f"{availability['availability_source'].eq('VERIFIED_OBSERVED').sum()}, "
        "fallback="
        f"{availability['availability_source'].eq('ANALYTICAL_FALLBACK').sum()}"
    ),
)
add_check(
    "final_modelling_split_unassigned",
    availability[
        "final_modelling_split"
    ].eq("UNASSIGNED").all(),
    "split assignment deferred",
)
add_check(
    "official_source_urls_recorded",
    all(
        source["url"].startswith("https://")
        for source in OFFICIAL_SOURCES.values()
    ),
    "HKO and GovHK provenance",
)

integrity = pd.DataFrame(check_rows)

if not integrity["passed"].all():
    raise AssertionError(
        "18tA blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "event_date",
        "detail",
        "blocking",
    ]
)

print("18tA integrity checks: PASS")
display(integrity)

18tA integrity checks: PASS


,check,passed,detail,blocking
0,verified_18s_input_pass,True,PASS,True
1,availability_rows_103,True,rows=103,True
2,availability_dates_unique,True,one row per frozen settlement date,True
3,fallback_dates_are_operational,True,Monday-Friday and not gazetted holiday,True
4,fallback_dates_follow_event_dates,True,strictly later calendar date,True
5,canonical_availability_after_event_end,True,no same-day outcome availability,True
6,availability_sources_exhaustive,True,"observed if verified, otherwise fallback",True
7,observed_plus_fallback_equals_103,True,"observed=0, fallback=103",True
8,final_modelling_split_unassigned,True,split assignment deferred,True
9,official_source_urls_recorded,True,HKO and GovHK provenance,True


In [7]:
output_availability = availability.copy()
output_calendar = calendar.copy()
output_evidence = evidence.copy()
output_holidays = holiday_table.copy()

for frame in [
    output_availability,
    output_calendar,
    output_evidence,
    output_holidays,
]:
    for column in frame.columns:
        if "date" in column.lower():
            if pd.api.types.is_datetime64_any_dtype(
                frame[column]
            ):
                frame[column] = frame[column].dt.strftime(
                    "%Y-%m-%d"
                )
        if "timestamp" in column.lower() or column.lower().endswith(
            "_hkt"
        ) or column.lower().endswith("_utc"):
            frame[column] = frame[column].astype("string")

availability_path = (
    OUT_DIR
    / "18tA_hko_publication_availability_panel.csv"
)
calendar_path = (
    OUT_DIR / "18tA_operational_calendar.csv"
)
evidence_path = (
    OUT_DIR
    / "18tA_verified_timestamp_evidence_inventory.csv"
)
holiday_path = (
    OUT_DIR / "18tA_gazetted_holiday_calendar.csv"
)
template_path = (
    OUT_DIR
    / "18tA_verified_publication_timestamp_template.csv"
)
integrity_path = (
    OUT_DIR / "18tA_integrity_checks.csv"
)
issues_path = OUT_DIR / "18tA_issues.csv"

output_availability.to_csv(
    availability_path,
    index=False,
)
output_calendar.to_csv(
    calendar_path,
    index=False,
)
output_evidence.to_csv(
    evidence_path,
    index=False,
)
output_holidays.to_csv(
    holiday_path,
    index=False,
)

template = output_availability[
    ["event_date", "sample_block"]
].copy()
template[
    "verified_publication_timestamp_hkt"
] = ""
template["verified"] = ""
template["evidence_reference"] = ""
template["evidence_note"] = ""
template.to_csv(template_path, index=False)

integrity.to_csv(integrity_path, index=False)
issues.to_csv(issues_path, index=False)

source_inventory_rows = [
    {
        "input_role": "18s_date_inventory",
        "path": str(
            DATE_INVENTORY_PATH.relative_to(ROOT)
        ),
        "rows": len(date_inventory),
        "sha256": sha256_file(DATE_INVENTORY_PATH),
    },
    {
        "input_role": "18s_target_panel",
        "path": str(TARGET_PATH.relative_to(ROOT)),
        "rows": len(target),
        "sha256": sha256_file(TARGET_PATH),
    },
    {
        "input_role": "18s_summary",
        "path": str(
            INPUT_SUMMARY_PATH.relative_to(ROOT)
        ),
        "rows": 1,
        "sha256": sha256_file(INPUT_SUMMARY_PATH),
    },
]

if MANUAL_EVIDENCE_PATH.is_file():
    source_inventory_rows.append(
        {
            "input_role": (
                "verified_observed_publication_evidence"
            ),
            "path": str(
                MANUAL_EVIDENCE_PATH.relative_to(ROOT)
            ),
            "rows": len(evidence),
            "sha256": sha256_file(
                MANUAL_EVIDENCE_PATH
            ),
        }
    )

for source_key, source in OFFICIAL_SOURCES.items():
    source_inventory_rows.append(
        {
            "input_role": source_key,
            "path": source["url"],
            "rows": 1,
            "sha256": "",
        }
    )

source_inventory = pd.DataFrame(
    source_inventory_rows
)
source_inventory_path = (
    OUT_DIR / "18tA_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

observed_count = int(
    availability[
        "availability_source"
    ].eq("VERIFIED_OBSERVED").sum()
)
fallback_count = int(
    availability[
        "availability_source"
    ].eq("ANALYTICAL_FALLBACK").sum()
)

summary = {
    "step": STEP,
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "settlement_dates": int(len(availability)),
    "verified_observed_timestamp_rows": observed_count,
    "analytical_fallback_rows": fallback_count,
    "evidence_file_status": evidence_file_status,
    "fallback_time_hkt": "14:00:00",
    "operational_working_day_definition": (
        "Monday-Friday excluding gazetted Hong Kong "
        "general holidays"
    ),
    "availability_convention": (
        "verified observed timestamp where supplied; "
        "otherwise analytical fallback"
    ),
    "fallback_is_exact_publication_claim": False,
    "earliest_event_date": str(
        availability["event_date"].min().date()
    ),
    "latest_event_date": str(
        availability["event_date"].max().date()
    ),
    "latest_canonical_availability_hkt": str(
        availability[
            "hko_publication_available_hkt"
        ].max()
    ),
    "maximum_calendar_days_after_event": int(
        availability[
            "availability_calendar_days_after_event"
        ].max()
    ),
    "dates_crossing_weekend": int(
        availability[
            "weekend_dates_crossed"
        ].fillna("").ne("").sum()
    ),
    "dates_crossing_gazetted_holiday": int(
        availability[
            "holiday_dates_crossed"
        ].fillna("").ne("").sum()
    ),
    "final_modelling_split_assigned": False,
    "probability_bridge_retained": False,
    "issue_rows": 0,
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(len(integrity)),
    "official_sources": OFFICIAL_SOURCES,
}

summary_path = OUT_DIR / "18tA_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "timezone": "Asia/Hong_Kong",
    "revision": "v1",
}
environment_path = (
    OUT_DIR / "18tA_environment.json"
)
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

report_lines = [
    "# 18tA canonical HKO publication availability",
    "",
    "**PASS**",
    "",
    "## Convention",
    "",
    (
        "A verified observed publication timestamp is used "
        "when supplied with evidence. Otherwise the analytical "
        "embargo is 14:00 HKT on the first Monday-Friday after "
        "settlement that is not a gazetted Hong Kong general "
        "holiday."
    ),
    "",
    (
        "The fallback is not represented as an exact-publication "
        "claim."
    ),
    "",
    "## Counts",
    "",
    f"- Settlement dates: {len(availability):,}",
    (
        "- Verified observed publication timestamps: "
        f"{observed_count:,}"
    ),
    (
        "- Analytical fallback timestamps: "
        f"{fallback_count:,}"
    ),
    (
        "- Dates whose fallback crosses a weekend: "
        f"{summary['dates_crossing_weekend']:,}"
    ),
    (
        "- Dates whose fallback crosses a gazetted holiday: "
        f"{summary['dates_crossing_gazetted_holiday']:,}"
    ),
    "",
    "## Official basis",
    "",
    (
        "- HKO Climatological Information Services: data are "
        "updated every working day before 2 p.m., up to the "
        "previous day."
    ),
    (
        "- GovHK 2026 general holidays: gazetted holiday dates "
        "used in the analytical operational calendar."
    ),
    "",
    "## Methodological boundary",
    "",
    (
        "No final modelling split is assigned. This panel only "
        "freezes the outcome-information time used by 18tB."
    ),
]

report_path = (
    REPORT_DIR
    / "18tA_hko_publication_availability_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []
for root in [OUT_DIR, REPORT_DIR]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18tA_sha256_manifest.csv":
            continue
        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = (
    OUT_DIR / "18tA_sha256_manifest.csv"
)
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print("18tA canonical availability release: PASS")

{
  "step": "18tA",
  "generated_at_utc": "2026-07-21T21:17:40.584891+00:00",
  "verdict": "PASS",
  "settlement_dates": 103,
  "verified_observed_timestamp_rows": 0,
  "analytical_fallback_rows": 103,
  "evidence_file_status": "ABSENT",
  "fallback_time_hkt": "14:00:00",
  "operational_working_day_definition": "Monday-Friday excluding gazetted Hong Kong general holidays",
  "availability_convention": "verified observed timestamp where supplied; otherwise analytical fallback",
  "fallback_is_exact_publication_claim": false,
  "earliest_event_date": "2026-03-16",
  "latest_event_date": "2026-06-30",
  "latest_canonical_availability_hkt": "2026-07-02 14:00:00+08:00",
  "maximum_calendar_days_after_event": 6,
  "dates_crossing_weekend": 32,
  "dates_crossing_gazetted_holiday": 11,
  "final_modelling_split_assigned": false,
  "probability_bridge_retained": false,
  "issue_rows": 0,
  "integrity_checks_passed": 10,
  "integrity_checks_total": 10,
  "official_sources": {
    "hko_update_conv